# SUMMER SCHOOL PROJECT 1 SUBMISSION (PART B)
* *NAME= SHYLA VIJAY***
* *ROLL NO.= 24/A06/064***
* *EMAIL ID= shyla7611@gmail.com***
* *CONTACT= 7982691483***

# README
1. Dataset Loading & Preprocessing
* Loaded SAR and EO image tensors (.pt files).
* Selected EO bands as per required usecase for each part.
* Converted tensors to [C, H, W] format.
* Added data augmentation: random flips and rotations to improve model generalization.

2. Model Architecture
* Implemented a CycleGAN framework using PyTorch.
* Built a ResNet-based generator with 6 residual blocks.
* Built a PatchGAN discriminator for classifying image patches as real or fake.
* Initialized weights using a normal distribution and used instance normalization.
* Used Instance Normalization and Tanh activation in the generator output.

3. Training Pipeline
* Defined a custom PyTorch dataset class (PairedTensorDataset_NIR_SWIR_RE) for loading paired SAR–EO samples.
* Split data into training and testing sets (80/20).
* Applied augmentations only on training data.
* Used PyTorch DataLoader for efficient mini-batch training.
* Calculated global min/max statistics for SAR and EO images.

4. Loss Functions
* Combined Adversarial Loss with Structural Similarity Index (SSIM) Loss to guide the generator.
* SSIM Loss helped improve the perceptual quality and structure preservation of the generated EO images.

5. Evaluation & Outputs
* Generated translated EO images from SAR inputs using the trained generator.
* Saved sample translated images in a folder (generated_samples/) for visual comparison.

Tools Used
PyTorch, torchvision, NumPy, Matplotlib

Environment: Kaggle Notebooks



In [25]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import random

class PairedTensorDataset_NIR_SWIR_RE(Dataset):
    def __init__(self, data_dir, augment=False):
        self.data_dir = data_dir
        self.augment = augment
        self.pair_ids = sorted(set(f.split('_')[0] for f in os.listdir(data_dir) if f.endswith('_sar.pt')))

    def __len__(self):
        return len(self.pair_ids)

    def __getitem__(self, idx):
        pair_id = self.pair_ids[idx]

        sar = torch.load(os.path.join(self.data_dir, f"{pair_id}_sar.pt"))  # [H, W, 2]
        eo  = torch.load(os.path.join(self.data_dir, f"{pair_id}_eo.pt"))   # [H, W, 13]

        # Rearrange to [C, H, W]
        if sar.shape[-1] == 2:
            sar = sar.permute(2, 0, 1)
        if eo.shape[-1] == 13:
            eo = eo.permute(2, 0, 1)

        # Select bands: B8 (7), B11 (10), B5 (4) → [3, H, W]
        eo_selected = torch.stack([eo[7], eo[10], eo[4]], dim=0)

        # --- Optional Data Augmentation ---
        if self.augment:
            if random.random() > 0.5:
                sar = torch.flip(sar, dims=[1])
                eo_selected = torch.flip(eo_selected, dims=[1])
            if random.random() > 0.5:
                sar = torch.flip(sar, dims=[2])
                eo_selected = torch.flip(eo_selected, dims=[2])
            if random.random() > 0.5:
                k = random.choice([1, 2, 3])
                sar = sar.rot90(k, dims=[1, 2])
                eo_selected = eo_selected.rot90(k, dims=[1, 2])

        return sar, eo_selected


print("Block done")

Block done


In [26]:
from torch.utils.data import DataLoader, random_split

# Dataset Path
DATA_DIR = '/kaggle/input/sar2eo-images/processed'
full_dataset = PairedTensorDataset_NIR_SWIR_RE(DATA_DIR, augment=True)

# Create dataset for part b
dataset = PairedTensorDataset_NIR_SWIR_RE(DATA_DIR)

sar = torch.load('/kaggle/input/sar2eo-images/processed/0000_sar.pt')
print(sar.min(), sar.max())
sar_min, sar_max = float('inf'), float('-inf')
eo_min, eo_max = float('inf'), float('-inf')

for sar, eo in dataset:
    sar_min = min(sar_min, sar.min().item())
    sar_max = max(sar_max, sar.max().item())
    eo_min = min(eo_min, eo.min().item())
    eo_max = max(eo_max, eo.max().item())

print(f"\n== Global Stats ==")
print(f"SAR: [{sar_min:.2f}, {sar_max:.2f}]")
print(f"EO:  [{eo_min:.2f}, {eo_max:.2f}]")

# Train/test split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
# Enable augmentation for training
train_dataset.dataset.augment = True
test_dataset.dataset.augment = False

# Dataloaders
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1)

# === Dataset Info ===
print(f"Total image pairs (SAR + EO-NIR/SWIR/RE): {len(dataset)}")
sar_sample, eo_sample = dataset[0]
print(f"SAR shape: {sar_sample.shape}, dtype: {sar_sample.dtype}")
print(f"EO shape:  {eo_sample.shape}, dtype: {eo_sample.dtype}")
print(f"SAR range: [{sar_sample.min():.2f}, {sar_sample.max():.2f}]")
print(f"EO range:  [{eo_sample.min():.2f}, {eo_sample.max():.2f}]")

# Batch test
batch_sar, batch_eo = next(iter(train_loader))
print(f"\nOne batch:")
print(f" SAR batch shape: {batch_sar.shape}")
print(f" EO  batch shape:  {batch_eo.shape}")


tensor(-1.) tensor(1.)

== Global Stats ==
SAR: [-1.00, 1.00]
EO:  [-1.00, 0.62]
Total image pairs (SAR + EO-NIR/SWIR/RE): 784
SAR shape: torch.Size([2, 256, 256]), dtype: torch.float32
EO shape:  torch.Size([3, 256, 256]), dtype: torch.float32
SAR range: [-1.00, 1.00]
EO range:  [-1.00, 0.45]

One batch:
 SAR batch shape: torch.Size([8, 2, 256, 256])
 EO  batch shape:  torch.Size([8, 3, 256, 256])


In [27]:
import torch.nn as nn

def init_weights(net, init_type='normal', gain=0.02):
    def init_func(m):  # initialize the network weights
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            nn.init.normal_(m.weight.data, 0.0, gain)
        if hasattr(m, 'bias') and m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    net.apply(init_func)
    return net

class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, input_nc, output_nc, n_blocks=6):
        super().__init__()
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(True),

            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(True)
        ]

        for _ in range(n_blocks):
            model += [ResnetBlock(256)]

        model += [
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),

            nn.ReflectionPad2d(3),
            nn.Conv2d(64, output_nc, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, input_nc):
        super().__init__()
        model = [
            nn.Conv2d(input_nc, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(256, 512, 4, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(512, 1, 4, padding=1)
        ]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)
print("Block done")

Block done


In [28]:
import torch 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G_AB = Generator(input_nc=2, output_nc=3).to(device)  # SAR → EO
G_BA = Generator(input_nc=3, output_nc=2).to(device)  # EO → SAR

D_A = Discriminator(input_nc=2).to(device)  # Discriminates SAR
D_B = Discriminator(input_nc=3).to(device)  # Discriminates EO


G_AB = init_weights(G_AB)
G_BA = init_weights(G_BA)
D_A = init_weights(D_A)
D_B = init_weights(D_B)
print ("Block completed")


Block completed


In [40]:
!pip install pytorch-msssim
from pytorch_msssim import ssim
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

lambda_ssim = 1.0  # Tune as needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [41]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

# === Setup ===
CHECKPOINT_DIR = "/kaggle/working/checkpoints_b"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
LATEST_CKPT = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")
START_EPOCH = 0
num_epochs = 10
lambda_cycle = 10.0

# === Move models to device ===
G_AB = G_AB.to(device)
G_BA = G_BA.to(device)
D_A = D_A.to(device)
D_B = D_B.to(device)

# === Optimizers ===
optimizer_G = optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()),
    lr=0.0002, betas=(0.5, 0.999)
)
optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))

# === Loss functions ===
criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss() 

# === Training Loop ===
saved_epochs = []  # track epochs for visualization

for epoch in range(START_EPOCH, num_epochs+1):
    G_AB.train()
    G_BA.train()

    for i, batch in enumerate(train_loader):
        real_A, real_B = batch
        real_A = real_A.to(device)
        real_B = real_B.to(device)

        # === Forward ===
        fake_B = G_AB(real_A)
        rec_A = G_BA(fake_B)

        fake_A = G_BA(real_B)
        rec_B = G_AB(fake_A)

        # === Generator Losses ===
        pred_fake_B = D_B(fake_B)
        pred_fake_A = D_A(fake_A)

        loss_GAN_AB = criterion_GAN(pred_fake_B, torch.ones_like(pred_fake_B))
        loss_GAN_BA = criterion_GAN(pred_fake_A, torch.ones_like(pred_fake_A))
        loss_cycle_A = criterion_cycle(rec_A, real_A)
        loss_cycle_B = criterion_cycle(rec_B, real_B)

        # SSIM loss (maximize similarity, so use 1 - ssim)
        loss_ssim_A = 1 - ssim(fake_A, real_A, data_range=2.0, size_average=True)
        loss_ssim_B = 1 - ssim(fake_B, real_B, data_range=2.0, size_average=True)

        loss_G = (
            loss_GAN_AB + loss_GAN_BA +
            lambda_cycle * (loss_cycle_A + loss_cycle_B) +
            lambda_ssim * (loss_ssim_A + loss_ssim_B)
        )

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        # === Discriminator A ===
        pred_real_A = D_A(real_A)
        pred_fake_A = D_A(fake_A.detach())
        loss_D_A = 0.5 * (
            criterion_GAN(pred_real_A, torch.ones_like(pred_real_A)) +
            criterion_GAN(pred_fake_A, torch.zeros_like(pred_fake_A))
        )
        optimizer_D_A.zero_grad()
        loss_D_A.backward()
        optimizer_D_A.step()

        # === Discriminator B ===
        pred_real_B = D_B(real_B)
        pred_fake_B = D_B(fake_B.detach())
        loss_D_B = 0.5 * (
            criterion_GAN(pred_real_B, torch.ones_like(pred_real_B)) +
            criterion_GAN(pred_fake_B, torch.zeros_like(pred_fake_B))
        )
        optimizer_D_B.zero_grad()
        loss_D_B.backward()
        optimizer_D_B.step()

        if i % 100 == 0:
            print(f"[Epoch {epoch}/{num_epochs}] [Batch {i}/{len(train_loader)}] "
                  f"Loss_G: {loss_G.item():.4f}, D_A: {loss_D_A.item():.4f}, D_B: {loss_D_B.item():.4f}")

    # === Save Checkpoint ===
    checkpoint_data = {
        'epoch': epoch,
        'G_AB': G_AB.state_dict(),
        'G_BA': G_BA.state_dict(),
        'D_A': D_A.state_dict(),
        'D_B': D_B.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D_A': optimizer_D_A.state_dict(),
        'optimizer_D_B': optimizer_D_B.state_dict(),
    }
    torch.save(checkpoint_data, os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch}.pth"))
    torch.save(checkpoint_data, LATEST_CKPT)
    print(f" Saved checkpoint for epoch {epoch}.")


[Epoch 0/10] [Batch 0/79] Loss_G: 2.6039, D_A: 0.2654, D_B: 0.2391
 Saved checkpoint for epoch 0.
[Epoch 1/10] [Batch 0/79] Loss_G: 2.6285, D_A: 0.2800, D_B: 0.2247
 Saved checkpoint for epoch 1.
[Epoch 2/10] [Batch 0/79] Loss_G: 2.6603, D_A: 0.2425, D_B: 0.2406
 Saved checkpoint for epoch 2.
[Epoch 3/10] [Batch 0/79] Loss_G: 2.5390, D_A: 0.2950, D_B: 0.2287
 Saved checkpoint for epoch 3.
[Epoch 4/10] [Batch 0/79] Loss_G: 2.6075, D_A: 0.2460, D_B: 0.2154
 Saved checkpoint for epoch 4.
[Epoch 5/10] [Batch 0/79] Loss_G: 2.7130, D_A: 0.2562, D_B: 0.2310
 Saved checkpoint for epoch 5.
[Epoch 6/10] [Batch 0/79] Loss_G: 2.6812, D_A: 0.2192, D_B: 0.2101
 Saved checkpoint for epoch 6.
[Epoch 7/10] [Batch 0/79] Loss_G: 2.4959, D_A: 0.2574, D_B: 0.2570
 Saved checkpoint for epoch 7.
[Epoch 8/10] [Batch 0/79] Loss_G: 2.5589, D_A: 0.2407, D_B: 0.2679
 Saved checkpoint for epoch 8.
[Epoch 9/10] [Batch 0/79] Loss_G: 2.9298, D_A: 0.2234, D_B: 0.2121
 Saved checkpoint for epoch 9.
[Epoch 10/10] [Batch

In [64]:
import os
import torch
import matplotlib.pyplot as plt

def save_outputs_separated(generator, test_loader, device, output_dir, num_samples=5):
    os.makedirs(output_dir, exist_ok=True)
    generator.eval()
    count = 0

    def norm(x): return (x + 1) / 2.0  # [-1, 1] → [0, 1]

    with torch.no_grad():
        for real_sar, real_eo in test_loader:
            real_sar = real_sar.to(device)
            real_eo = real_eo.to(device)
            fake_eo = generator(real_sar)

            for i in range(real_sar.size(0)):
                if count >= num_samples:
                    break

                # === Normalize ===
                sar_img = norm(real_sar[i, 0].cpu())
                fake_img = norm(fake_eo[i].cpu())
                real_img = norm(real_eo[i].cpu())

                # === Create figure ===
                fig, axes = plt.subplots(1, 3, figsize=(16, 6), dpi=100)
                fig.subplots_adjust(wspace=0.25, left=0.05, right=0.95, top=0.88, bottom=0.12)
                fig.suptitle("SAR → Fake EO → Real EO", fontsize=16)
                fig.patch.set_facecolor('white')

                # Plot SAR
                axes[0].imshow(sar_img.numpy(), cmap='gray')
                axes[0].set_title("SAR Input")
                axes[0].axis('off')

                # Plot Fake EO
                axes[1].imshow(fake_img.permute(1, 2, 0).numpy())
                axes[1].set_title("Fake EO")
                axes[1].axis('off')

                # Plot Real EO
                axes[2].imshow(real_img.permute(1, 2, 0).numpy())
                axes[2].set_title("Real EO")
                axes[2].axis('off')

                # === Save figure ===
                save_path = os.path.join(output_dir, f"sample_{count + 1}.png")
                plt.savefig(save_path)
                plt.close()
                print(f"[Saved] {save_path}")

                count += 1

            if count >= num_samples:
                break


In [65]:
# Load checkpoint (if not already loaded)
checkpoint = torch.load(LATEST_CKPT, map_location=device)
G_AB.load_state_dict(checkpoint['G_AB'])

# Save images
OUTPUT_DIR = "/kaggle/working/eo_triplet_vis"
save_outputs_separated(G_AB, test_loader, device, output_dir=OUTPUT_DIR, num_samples=5)


[Saved] /kaggle/working/eo_triplet_vis/sample_1.png
[Saved] /kaggle/working/eo_triplet_vis/sample_2.png
[Saved] /kaggle/working/eo_triplet_vis/sample_3.png
[Saved] /kaggle/working/eo_triplet_vis/sample_4.png
[Saved] /kaggle/working/eo_triplet_vis/sample_5.png


In [43]:
import torch
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# --- Helper Functions ---
def tensor_to_numpy(img_tensor):
    img = img_tensor.detach().cpu() * 0.5 + 0.5  # De-normalize [-1,1] → [0,1]
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)

def compute_ssim(img1, img2):
    img1_np = tensor_to_numpy(img1)
    img2_np = tensor_to_numpy(img2)
    return np.mean([
        ssim(img1_np[:, :, c], img2_np[:, :, c], data_range=1.0)
        for c in range(img1_np.shape[2])
    ])

def compute_psnr(img1, img2):
    return psnr(tensor_to_numpy(img1), tensor_to_numpy(img2), data_range=1.0)

def compute_mae(img1, img2):
    return torch.mean(torch.abs(img1 - img2)).item()

def compute_ndvi_difference(fake_img, real_img):
    fake = tensor_to_numpy(fake_img)
    real = tensor_to_numpy(real_img)
    fake_ndvi = (fake[:, :, 0] - fake[:, :, 2]) / (fake[:, :, 0] + fake[:, :, 2] + 1e-8)
    real_ndvi = (real[:, :, 0] - real[:, :, 2]) / (real[:, :, 0] + real[:, :, 2] + 1e-8)
    return np.mean(np.abs(fake_ndvi - real_ndvi))

# --- Evaluation Function ---
def evaluate_generator(generator, data_loader, device, max_batches=None):
    generator.eval()
    ssim_scores, psnr_scores, mae_scores, ndvi_scores = [], [], [], []

    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            real_A, real_B = batch
            real_A = real_A.to(device)
            real_B = real_B.to(device)
            fake_B = generator(real_A)

            for j in range(real_A.size(0)):
                ssim_scores.append(compute_ssim(fake_B[j], real_B[j]))
                psnr_scores.append(compute_psnr(fake_B[j], real_B[j]))
                mae_scores.append(compute_mae(fake_B[j], real_B[j]))
                ndvi_scores.append(compute_ndvi_difference(fake_B[j], real_B[j]))

            if max_batches is not None and (i + 1) >= max_batches:
                break

    metrics_result = {
        "Mean SSIM": np.mean(ssim_scores),
        "Mean PSNR": np.mean(psnr_scores),
        "Mean MAE": np.mean(mae_scores),
        "Mean NDVI Difference": np.mean(ndvi_scores)
    }

    print("\nEvaluation Metrics on Test Set:")
    for key, val in metrics_result.items():
        print(f" - {key}: {val:.4f}")

    return metrics_result

# Evaluate on test_loader
metrics = evaluate_generator(G_AB, test_loader, device)



Evaluation Metrics on Test Set:
 - Mean SSIM: 0.9118
 - Mean PSNR: 30.3655
 - Mean MAE: 0.0404
 - Mean NDVI Difference: 0.0080


In [66]:
# Evaluate on test_loader
metrics = evaluate_generator(G_AB, test_loader, device)

# Save metrics to CSV
import pandas as pd
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv("/kaggle/working/metrics_PARTB.csv", index=False)
print(" Metrics saved to /kaggle/working/metrics_PARTB.csv")


Evaluation Metrics on Test Set:
 - Mean SSIM: 0.9118
 - Mean PSNR: 30.3655
 - Mean MAE: 0.0404
 - Mean NDVI Difference: 0.0080
 Metrics saved to /kaggle/working/metrics_PARTB.csv
